In [3]:
# Import libraries for data manipulation
import pandas as pd
import numpy as np

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Import machine learning tools
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Import evaluation metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

# Library to save or load trained model
import pickle

In [4]:
# Load dataset

data = pd.read_csv("microfinance_final_project_dataset.csv")

# Display first 5 rows
data.head()

,applicant_id,age,gender,region,education,employment,employment_years,monthly_income,loan_amount,loan_duration,interest_rate,loan_purpose,credit_score,dependents,previous_loans,repayment_history,collateral,debt_to_income_ratio,loan_status
0,100000,30,Female,Southeast Asia,Secondary,Farmer,4,17781,15506,36,22.61,Medical,456,4,3,Good,No,0.073,0
1,100001,36,Female,Southeast Asia,Postgraduate,Self-employed,18,7571,30852,24,17.88,Agriculture,795,3,4,Poor,Yes,0.340,0
2,100002,46,Male,Latin America,Graduate,Small Business,17,6762,101174,12,23.14,Medical,593,3,2,Average,Yes,1.247,0
3,100003,41,Male,Southeast Asia,Primary,Small Business,11,34389,22645,36,25.71,Agriculture,442,0,1,Good,Yes,0.055,0
4,100004,30,Male,Sub-Saharan Africa,Secondary,Self-employed,28,34942,149135,36,22.83,Housing,548,0,0,Good,Yes,0.356,1


In [5]:
X = data.drop("loan_status", axis=1)   # input features
y = data["loan_status"]              

In [6]:
X = pd.get_dummies(X, drop_first=True)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [8]:
model = RandomForestClassifier(random_state=42)

# Train model on training data

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [9]:
y_pred = model.predict(X_test)

# Predict probability of default

y_prob = model.predict_proba(X_test)[:,1]

In [10]:
# Predict risk for a single applicant using only credit score

# Create a row with average values for all features
new_person = X.mean().to_frame().T

# Change only the credit score
new_person["credit_score"] = 500   # you can change this value

# Predict risk
prediction = model.predict(new_person)
probability = model.predict_proba(new_person)[:,1]

print("Credit Score:", new_person["credit_score"].values[0])
print("Default Probability:", probability[0])

if prediction[0] == 1:
    print("Risk Level: HIGH RISK")
else:
    print("Risk Level: LOW RISK")

Credit Score: 500
Default Probability: 0.27
Risk Level: LOW RISK


In [11]:
from scipy.stats import ks_2samp

# Separate probabilities for each class
prob_default = y_prob[y_test == 1]
prob_non_default = y_prob[y_test == 0]

# Compute KS statistic
ks_statistic = ks_2samp(prob_default, prob_non_default)

print("KS Statistic:", ks_statistic.statistic)

KS Statistic: 0.34761009667024706
